# VS Code + AgentCore Gateway: Serverless OAuth Proxy

## Overview

This notebook deploys a **serverless OAuth proxy** using API Gateway + Lambda,
eliminating the need for developers to run local proxy and callback servers.

### Architecture

![Serverless Proxy Architecture](generated-diagrams/vscode-agentcore-serverless-proxy.png)

### What This Deploys

1. **API Gateway** - Public endpoint for VS Code to connect to
2. **MCP Proxy Lambda** - OAuth metadata, callback interception, token proxying, MCP forwarding
3. **3LO Callback Lambda** - Outbound OAuth callbacks, CompleteResourceTokenAuth
4. **Cognito User Pool** - JWT tokens for inbound authentication
5. **AgentCore Gateway** - MCP server with Confluence target

## Step 1: Setup

In [ ]:
# Install dependencies
!pip3 install -r requirements.txt --quiet

Deploy the CDK stack:

```bash
cd cdk
npm install
cdk deploy
```

then copy the output in the cell below replacing the content.

In [ ]:
import boto3

In [ ]:
output = """
<CDK output content>
"""

In [ ]:
config = {}
for el in output.split("\n"):
    if "=" in el:
        key, value = el.split("=", 1)
        config[key.strip().replace("CdkStack.", "")] = value.strip()

## Step 1: Create Cognito User

In [ ]:
COGNITO_USERNAME = "vscode-user@example.com"
COGNITO_PASSWORD = "TempPassword123!"

cognito = boto3.client("cognito-idp")
user_pool_id = config["UserPoolId"]
try:
    cognito.admin_create_user(
        UserPoolId=user_pool_id,
        Username=COGNITO_USERNAME,
        TemporaryPassword=COGNITO_PASSWORD,
        MessageAction='SUPPRESS',
        UserAttributes=[{'Name': 'email', 'Value': f'{COGNITO_USERNAME}'},
                       {'Name': 'email_verified', 'Value': 'true'}]
    )
    cognito.admin_set_user_password(
        UserPoolId=user_pool_id, Username=COGNITO_USERNAME,
        Password=COGNITO_PASSWORD, Permanent=True
    )
    print(f"✓ User created: {COGNITO_USERNAME}")
except cognito.exceptions.UsernameExistsException:
    print(f"✓ User exists: {COGNITO_USERNAME}")

## Step 2: Enter Atlassian OAuth Credentials

Copy the `.env.template` file to an `.env` file and fill in the required information from your Atlassian app.
Execute the following cell to load the values in the notebook.

In [ ]:
import dotenv
import os

dotenv.load_dotenv(override=True)

ATLASSIAN_CLIENT_ID = os.environ.get("ATLASSIAN_CLIENT_ID", None)
ATLASSIAN_CLIENT_SECRET = os.environ.get("ATLASSIAN_CLIENT_SECRET", None)
ATLASSIAN_TENANT = os.environ.get("ATLASSIAN_TENANT", None)

if ATLASSIAN_CLIENT_ID is None:
    print("⚠️  Replace placeholder values with your Atlassian OAuth credentials")
else:
    print("✓ Atlassian credentials configured")

## Step 3: Create Atlassian Credential Provider

In [ ]:
import boto3

client = boto3.client('bedrock-agentcore-control')

In [ ]:
credential_provider_name = "Atlassian"
atlassian_provider = client.create_oauth2_credential_provider(
    name=credential_provider_name,
    credentialProviderVendor="AtlassianOauth2",
    oauth2ProviderConfigInput={
        'atlassianOauth2ProviderConfig': {
            'clientId': ATLASSIAN_CLIENT_ID,
            'clientSecret': ATLASSIAN_CLIENT_SECRET
        }
    }
)
credential_provider_arn = atlassian_provider['credentialProviderArn']
agentcore_callback_url = atlassian_provider['callbackUrl']

print(f"✓ Credential Provider: {credential_provider_arn}")
print(f"\n{'='*60}")
print("⚠️  Register this callback URL in your Atlassian app:")
print(f"   {agentcore_callback_url}")
print(f"{'='*60}")

## Step 4: Create Confluence Target

In [ ]:
import json
confluence_openapi_spec = {
    "openapi": "3.0.0",
    "info": {"title": "Confluence Cloud API", "version": "1.0.0"},
    "servers": [{"url": "https://api.atlassian.com/ex/confluence"}],
    "paths": {
        "/{cloudId}/wiki/api/v2/spaces": {
            "get": {
                "operationId": "getSpaces",
                "summary": "Get all Confluence spaces",
                "security": [{"BearerAuth": []}],
                "parameters": [
                    {"name": "cloudId", "in": "path", "required": True, "schema": {"type": "string"}},
                    {"name": "limit", "in": "query", "schema": {"type": "integer", "default": 25}}
                ],
                "responses": {"200": {"description": "List of spaces"}}
            }
        },
        "/{cloudId}/wiki/api/v2/pages": {
            "get": {
                "operationId": "getPages",
                "summary": "Get Confluence pages",
                "security": [{"BearerAuth": []}],
                "parameters": [
                    {"name": "cloudId", "in": "path", "required": True, "schema": {"type": "string"}},
                    {"name": "space-id", "in": "query", "schema": {"type": "string"}},
                    {"name": "limit", "in": "query", "schema": {"type": "integer", "default": 25}}
                ],
                "responses": {"200": {"description": "List of pages"}}
            }
        }
    },
    "components": {"securitySchemes": {"BearerAuth": {"type": "http", "scheme": "bearer"}}}
}

CONFLUENCE_SCOPES = ["read:space:confluence", "read:page:confluence", "read:confluence-content.all", "offline_access"]
api_endpoint = config["ApiEndpoint"]
gateway_id = config["Gateway"]
DEFAULT_RETURN_URL = f"{api_endpoint}/oauth2/callback"

target_response = client.create_gateway_target(
    name="Confluence",
    description="Confluence Cloud API with 3LO OAuth",
    gatewayIdentifier=gateway_id,
    credentialProviderConfigurations=[{
        "credentialProviderType": "OAUTH",
        "credentialProvider": {
            "oauthCredentialProvider": {
                "providerArn": credential_provider_arn,
                "grantType": "AUTHORIZATION_CODE",
                "defaultReturnUrl": DEFAULT_RETURN_URL,
                "scopes": CONFLUENCE_SCOPES
            }
        }
    }],
    targetConfiguration={"mcp": {"openApiSchema": {"inlinePayload": json.dumps(confluence_openapi_spec)}}}
)
target_id = target_response["targetId"]
print(f"✓ Confluence target: {target_id}")

## Step 5: Get Atlassian Cloud ID

In [ ]:
import requests
if ATLASSIAN_TENANT and not ATLASSIAN_TENANT.startswith("<"):
    try:
        url = f"https://{ATLASSIAN_TENANT}.atlassian.net/_edge/tenant_info"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        tenant_info = response.json()
        ATLASSIAN_CLOUD_ID = tenant_info.get("cloudId")
        print(f"✓ Cloud ID: {ATLASSIAN_CLOUD_ID}")
    except Exception as e:
        print(f"Could not retrieve Cloud ID: {e}")
        ATLASSIAN_CLOUD_ID = "<your-cloud-id>"
else:
    ATLASSIAN_CLOUD_ID = "<your-cloud-id>"
    print("⚠️  Set ATLASSIAN_TENANT to retrieve Cloud ID")

## Step 6: VS Code Configuration

No local servers needed! Just configure VS Code to point to the API Gateway.

----

## Cleanup (Optional)

Run this cell to delete all resources created by this notebook.

In [ ]:
def cleanup():
    """Delete all resources created by this notebook."""
    print("Cleaning up resources...")
    # Delete Gateway target and gateway
    try:
        client.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
        
    except: pass
    
    # Delete credential provider
    try:
        client.delete_oauth2_credential_provider(name=credential_provider_name)
        print(f"✓ Deleted Credential Provider")
    except: pass

# Uncomment to run cleanup:
# cleanup()

To remove the remaining resources run:

```bash
cd cdk
cdk destroy
```